# Stage 7 -- 通路富集分析 (Pathway Enrichment)

本 notebook 基于 per-cluster 差异表达基因做通路富集，包含两种互补方法：

1. **GSEApy Enrichr** -- 在线 ORA（Over-Representation Analysis），将每簇 top DEG
   与 Reactome / KEGG / GO Biological Process 数据库比对，找出显著富集的通路。
2. **decoupler 通路活性评分** -- 用 PROGENy（14 条信号通路）和 MSigDB Hallmark（50 条
   标志性通路）的基因集，通过 ULM（Univariate Linear Model）对每个细胞打分，然后按簇聚合。

## 为什么两种方法互补？

- **Enrichr / ORA**：局部视角——回答「这个簇的标记基因富集了哪些通路」，需要先定 top DEG 阈值，
  优点是直接给出通路名称和 p 值，PI 容易解读。局限是阈值依赖且忽略基因的效应量（logFC 大小）和方向。
- **decoupler 活性评分**：全局视角——回答「每个细胞/簇中，某条通路有多活跃」，直接在表达矩阵上
  计算通路活性分数，不依赖先验 DEG 列表。局限是依赖于通路基因集的完备性。

## 分组列

默认使用 `LEIDEN_COL` (leiden_res_0.6) 作为分组列——当前基线 `cell_type_final_v1`
全为 NaN，PI 完成真实注释后将 `CELL_TYPE_COL` 设为 `cell_type_final_v1` 即可切换到基于注释的通路分析。

产出：
- Enrichr 结果表 -> `results/tables/stage7_pathway_enrichr_*.csv`
- decoupler 通路活性 -> `results/tables/stage7_pathway_decoupler_*.csv`
- 可视化 -> `results/figures/stage7_pathway_*.png`
- checkpoint -> `OUTPUT_PATH`

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH           -- stage6 注释结果 h5ad（含 LEIDEN_COL）
# OUTPUT_PATH             -- 本 notebook 产出 checkpoint
# LEIDEN_COL              -- Leiden 簇标签 obs 列（默认分组）
# CELL_TYPE_COL           -- 细胞类型列（PI 有真实注释后可改）
#                            **重要提醒**：当前基线 cell_type_final_v1 全为 NaN，
#                            通路分析使用 LEIDEN_COL。PI 完成后设为 "cell_type_final_v1"。
# N_TOP_GENES_PER_CLUSTER -- 每簇取 top 标记基因数（enrichr 输入）
# GSEAPY_GENE_SETS        -- GSEApy enrichr 查询的 gene set 库列表
# GSEAPY_PVAL_CUTOFF      -- 富集显著性阈值
# GSEAPY_N_TOP_CLUSTERS   -- enrichr 分析的优先簇数（按 DEG 显著度排序，取前 N）
#                            设为 None 则分析全部簇
# ENRICHR_TIMEOUT         -- enrichr 单次查询超时秒数
# DECOUPLER_METHOD        -- decoupler 通路评分方法（ulm/mlm/gsva/viper）
# N_TOP_PATHWAYS_PLOT     -- 可视化展示的 top 通路数

UPSTREAM_PATH = "results/nancang_stage6_annotated_v1.h5ad"
OUTPUT_PATH   = "results/stage7_pathway.h5ad"

LEIDEN_COL    = "leiden_res_0.6"
CELL_TYPE_COL = "cell_type_final_v1"

N_TOP_GENES_PER_CLUSTER = 50

GSEAPY_GENE_SETS = [
    "KEGG_2021_Human",
    "Reactome_2022",
    "GO_Biological_Process_2023",
]
GSEAPY_PVAL_CUTOFF = 0.05
GSEAPY_N_TOP_CLUSTERS = None    # None = 全部簇；PI 可设 5 限制网络请求数

ENRICHR_TIMEOUT = 30            # 秒

DECOUPLER_METHOD = "ulm"       # ulm: 单变量线性模型，计算快、可解释、适合探索

N_TOP_PATHWAYS_PLOT = 10

In [ ]:
# 项目根目录定位 + sys.path。
# 多级回退：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/stage7/ 回退两级。
import sys, os, gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

In [ ]:
# 导入依赖。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import math as _math

# GSEApy 和 decoupler 是 conda scrna-integration 环境预装的依赖，
# 不在 framework 框架内部，notebook 直接 import。
import gseapy as gp
import decoupler as dc

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  gseapy {gp.__version__}  |  decoupler {dc.__version__}")

In [ ]:
# 加载上游 stage6 输出。
# 契约：需包含 LEIDEN_COL 和 counts layer（decoupler 需要原始计数）。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"layers: {list(adata.layers.keys())}")
print(f"LEIDEN_COL '{LEIDEN_COL}' 存在: {LEIDEN_COL in adata.obs.columns}")
if LEIDEN_COL in adata.obs.columns:
    print(f"  簇数: {adata.obs[LEIDEN_COL].nunique()}")
    print(f"  簇: {sorted(adata.obs[LEIDEN_COL].unique())}")
print(f"CELL_TYPE_COL '{CELL_TYPE_COL}' 存在: {CELL_TYPE_COL in adata.obs.columns}")
if CELL_TYPE_COL in adata.obs.columns:
    _n_valid = adata.obs[CELL_TYPE_COL].notna().sum()
    print(f"  有效注释数: {_n_valid} / {adata.n_obs}")
    if _n_valid > 0:
        print(f"  细胞类型: {sorted(adata.obs[CELL_TYPE_COL].dropna().unique())}")

## 0. Per-cluster DEG -- 差异表达（通路富集的上游输入）

用 `sc.tl.rank_genes_groups` 对每个分组（Leiden 簇或细胞类型）找标记基因。
**为什么用 Wilcoxon？** 非参数检验，不假设表达量服从正态分布，
对稀疏 scRNA-seq 数据鲁棒性好。「1-vs-rest」模式下每个簇与所有其他细胞比较。

这部分与 `deg.ipynb` 共享相同逻辑——如果上游 h5ad 已包含 `rank_genes_per_cluster`
结果则跳过计算直接复用；否则现场计算（通常在 30 秒内完成）。

In [ ]:
# 确定分组列：优先 CELL_TYPE_COL（若有效），其次 LEIDEN_COL。
_group_col = LEIDEN_COL
if CELL_TYPE_COL in adata.obs.columns:
    _n_valid = adata.obs[CELL_TYPE_COL].notna().sum()
    if _n_valid >= 10:
        _group_col = CELL_TYPE_COL
        print(f"使用细胞类型注释列: {CELL_TYPE_COL} ({_n_valid} 个有效细胞)")
    else:
        print(f"CELL_TYPE_COL '{CELL_TYPE_COL}' 有效细胞数 {_n_valid} < 10")
        print(f"  回退到 LEIDEN_COL: {LEIDEN_COL}")
else:
    print(f"CELL_TYPE_COL '{CELL_TYPE_COL}' 不存在")
    print(f"  使用 LEIDEN_COL: {LEIDEN_COL}")

In [ ]:
# Per-cluster 差异表达——每个簇 vs 所有其他簇。
# 如果 adata.uns 已有结果则跳过计算（复用 deg.ipynb 产出）。
_deg_key = "rank_genes_per_cluster"
if _deg_key in adata.uns:
    print(f"检测到已有 DEG 结果 '{_deg_key}'，跳过计算直接复用")
else:
    # 确保分组列是 category 或 str 类型
    if not hasattr(adata.obs[_group_col], "cat"):
        adata.obs[_group_col] = adata.obs[_group_col].astype(str)

    sc.tl.rank_genes_groups(
        adata,
        groupby=_group_col,
        method="wilcoxon",
        n_genes=N_TOP_GENES_PER_CLUSTER,
        key_added=_deg_key,
        use_raw=False,
    )
    print(f"per-cluster DEG 完成 (key='{_deg_key}', groupby='{_group_col}')")

# 提取每簇 top 标记基因 -> dict，供 enrichr 使用。
_cluster_genes = {}  # cluster_id -> list of top gene symbols
_cluster_ids = sorted(adata.obs[_group_col].astype(str).unique())
for _cid in _cluster_ids:
    try:
        _df = sc.get.rank_genes_groups_df(adata, group=_cid, key=_deg_key)
        _cluster_genes[_cid] = _df["names"].head(N_TOP_GENES_PER_CLUSTER).tolist()
    except Exception:
        _cluster_genes[_cid] = []

# 按 DEG 显著度排序簇：每个簇取 top 基因的 -log10(p_adj) 均值作为显著度代理。
_cluster_sig = {}
for _cid in _cluster_ids:
    try:
        _df = sc.get.rank_genes_groups_df(adata, group=_cid, key=_deg_key)
        _cluster_sig[_cid] = (-np.log10(_df["pvals_adj"].clip(lower=1e-300))).mean()
    except Exception:
        _cluster_sig[_cid] = 0.0
_cluster_order = sorted(_cluster_sig, key=_cluster_sig.get, reverse=True)
print(f"\n各簇 top {N_TOP_GENES_PER_CLUSTER} 标记基因已提取，共 {len(_cluster_genes)} 个分组")
print(f"簇显著度排序（均值 -log10(p_adj) 降序）: "
      f"{', '.join(f'{c}:{_cluster_sig[c]:.1f}' for c in _cluster_order[:5])}...")

## 网络守卫

GSEApy enrichr 和 decoupler `op.progeny()` / `op.hallmark()` 都需要联网获取。
本 cell 尝试一次轻量网络探测：用 HTTP GET 检测 Enrichr 服务器可达性。
如果网络不可用则优雅跳过所有在线分析，仅保留离线模式警告。

**为什么先守后卫？** 离线环境（集群计算节点 / 无 VPN 的本地）不应因网络
问题导致整个 notebook 崩溃——参考 stage2 SoupX 和 stage7 pseudotime Monocle3 守卫模式。

In [ ]:
# 网络守卫：探测 Enrichr API 可用性。
# 如果超时或连接失败，后续所有在线分析自动跳过。
_network_available = False
try:
    import requests as _requests
    _resp = _requests.get("https://maayanlab.cloud/Enrichr/", timeout=15)
    if _resp.status_code == 200:
        _network_available = True
        print("网络可用 -- Enrichr 和 decoupler 在线通路数据库可正常访问")
    else:
        print(f"Enrichr 服务器返回状态码 {_resp.status_code}，在线分析将跳过")
except Exception as _e:
    print(f"网络不可用: {type(_e).__name__}: {_e}")
    print("在线分析（enrichr / decoupler op.progeny / op.hallmark）将优雅跳过")
    print("离线模式：仅 decoupler 本地的通路活性评分可用（需预下载 gene set 文件）")

## 1. GSEApy Enrichr -- per-cluster ORA（在线）

对每簇的 top 标记基因做 Over-Representation Analysis，
与 Reactome / KEGG / GO Biological Process 比对。

**为什么用 Enrichr？** 基因集库最新（2021-2023 版本），覆盖 Reactome/KEGG/GO/TRRUST/TRANSFAC
等数百个库，一次查询同时出多条通路结果，适合「我有一组基因，想知道它们富集什么通路」的探索场景。

**为什么限制簇数？** 每个簇 x 每个 gene set 库 = N 次 HTTP 请求。
Enrichr API 未声明严格 rate limit 但大量并发请求会显著变慢。
默认取全部簇的 per-cluster 结果（少量簇时无影响），
PI 可在 PARAMS 中设 `GSEAPY_N_TOP_CLUSTERS` 限制。

In [ ]:
# GSEApy enrichr: per-cluster ORA。
# 每簇取 top DEG，对每个 gene set 库做一次 enrichr 查询。
_enrichr_results = {}  # (cluster_id, gene_set_lib) -> Enrichr result DataFrame

if _network_available:
    # 确定要分析的簇
    if GSEAPY_N_TOP_CLUSTERS and GSEAPY_N_TOP_CLUSTERS < len(_cluster_order):
        _clusters_to_run = _cluster_order[:GSEAPY_N_TOP_CLUSTERS]
        print(f"仅分析前 {GSEAPY_N_TOP_CLUSTERS} 个最显著簇: {_clusters_to_run}")
    else:
        _clusters_to_run = _cluster_order
        print(f"分析全部 {len(_clusters_to_run)} 个簇")

    _n_total = len(_clusters_to_run) * len(GSEAPY_GENE_SETS)
    _n_done = 0
    _n_failed = 0

    for _cid in _clusters_to_run:
        _genes = _cluster_genes.get(_cid, [])
        if len(_genes) < 5:
            print(f"  簇 {_cid}: 标记基因不足 5 个（{len(_genes)}），跳过 enrichr")
            continue

        for _gs_lib in GSEAPY_GENE_SETS:
            try:
                _enr = gp.enrichr(
                    gene_list=_genes,
                    gene_sets=_gs_lib,
                    organism="human",
                    outdir=None,
                    cutoff=GSEAPY_PVAL_CUTOFF,
                    no_plot=True,
                    verbose=False,
                )
                if _enr.results is not None and len(_enr.results) > 0:
                    _enrichr_results[(_cid, _gs_lib)] = _enr.results
                _n_done += 1
            except Exception as _e:
                _n_failed += 1
                if _n_failed <= 3:
                    print(f"  WARNING: 簇 {_cid} x {_gs_lib} enrichr 失败: "
                          f"{type(_e).__name__}: {str(_e)[:120]}")

    print(f"\nenrichr 完成: {_n_done}/{_n_total} 查询成功, {_n_failed} 失败")
    print(f"获得结果: {len(_enrichr_results)} 组")
else:
    _skip_msg = (
        "\n" + "=" * 60 + "\n"
        "GSEApy enrichr 需要联网，当前网络不可用，以下分析将优雅跳过。\n\n"
        "要启用 enrichr 分析，请:\n"
        "  1. 确认网络连接正常（可访问 maayanlab.cloud）\n"
        "  2. 重跑本 notebook\n"
        "=" * 60 + "\n"
    )
    print(_skip_msg)

In [ ]:
# 汇总 enrichr 结果：收束为一个总表。
_enrichr_all = []
for (_cid, _gs_lib), _df in _enrichr_results.items():
    _df_copy = _df.copy()
    _df_copy["cluster"] = _cid
    _df_copy["gene_set_library"] = _gs_lib
    _enrichr_all.append(_df_copy)

if _enrichr_all:
    _enrichr_df = pd.concat(_enrichr_all, ignore_index=True)
    _enrichr_csv = "results/tables/stage7_pathway_enrichr_all.csv"
    _enrichr_df.to_csv(_enrichr_csv, index=False)
    print(f"enrichr 汇总表已保存: {_enrichr_csv}  ({len(_enrichr_df)} 条)")

    # 简要展示：每个 gene set 库 top 5 通路
    for _gs_lib in GSEAPY_GENE_SETS:
        _sub = _enrichr_df[_enrichr_df["gene_set_library"] == _gs_lib]
        if len(_sub) > 0:
            print(f"\n{_gs_lib} -- top 5 通路（按 Adjusted P-value）:")
            _show = _sub.nsmallest(5, "Adjusted P-value")[
                ["cluster", "Term", "Adjusted P-value", "Overlap"]
            ]
            print(_show.to_string(index=False))
else:
    print("无 enrichr 结果")

### Enrichr 可视化 -- barplot top pathways

每簇取 top N 显著通路画水平条形图。
横轴=-log10(Adjusted P-value)，条形越长为越显著的通路。
**为什么画 barplot？** 一目了然展示「这个簇的生物功能与哪些通路关联最强」，
适合 PI 在组会中快速判断各簇的功能标签。

In [ ]:
# Enrichr barplot: 每簇在每个 gene set 库的 top 通路。
# 为什么一个簇一个 subplot？簇数量通常不多（10-20），单图可比较。
if _enrichr_all and _network_available:

    for _gs_lib in GSEAPY_GENE_SETS:
        # 收集该库所有簇的 top 通路
        _plot_records = []
        for _cid in _clusters_to_run:
            _key = (_cid, _gs_lib)
            if _key in _enrichr_results:
                _df = _enrichr_results[_key]
                _top = _df.nsmallest(N_TOP_PATHWAYS_PLOT, "Adjusted P-value")
                for _, _row in _top.iterrows():
                    _plot_records.append({
                        "cluster": _cid,
                        "pathway": _row["Term"],
                        "-log10(p_adj)": -np.log10(
                            max(_row["Adjusted P-value"], 1e-300)
                        ),
                    })

        if not _plot_records:
            continue

        _plot_df = pd.DataFrame(_plot_records)
        _n_clusters = _plot_df["cluster"].nunique()
        _n_cols = min(3, _n_clusters)
        _n_rows = _math.ceil(_n_clusters / _n_cols)

        fig, axes = plt.subplots(
            _n_rows, _n_cols,
            figsize=(6 * _n_cols, 4 * _n_rows),
        )
        if _n_rows * _n_cols == 1:
            axes = [axes]
        else:
            axes = axes.flatten()

        for _i, _cid in enumerate(sorted(_plot_df["cluster"].unique())):
            _ax = axes[_i]
            _sub = _plot_df[_plot_df["cluster"] == _cid].nlargest(
                N_TOP_PATHWAYS_PLOT, "-log10(p_adj)"
            )
            # 截断过长通路名
            _sub["label"] = _sub["pathway"].apply(
                lambda x: x[:50] + "..." if len(str(x)) > 50 else str(x)
            )
            _ax.barh(
                range(len(_sub)),
                _sub["-log10(p_adj)"].values[::-1],
                color="steelblue",
                height=0.7,
            )
            _ax.set_yticks(range(len(_sub)))
            _ax.set_yticklabels(_sub["label"].values[::-1], fontsize=7)
            _ax.set_xlabel("-log10(p_adj)")
            _ax.set_title(f"Cluster {_cid}: {_gs_lib}")
            _ax.axvline(-np.log10(GSEAPY_PVAL_CUTOFF), ls="--",
                        color="gray", lw=0.8)

        # 隐藏多余 subplot
        for _ax in axes[_n_clusters:]:
            _ax.set_visible(False)

        plt.tight_layout()
        _gs_short = _gs_lib.split("_")[0].lower()
        _fig_path = f"results/figures/stage7_pathway_enrichr_barplot_{_gs_short}.png"
        fig.savefig(_fig_path, dpi=200, bbox_inches="tight")
        plt.close("all")
        print(f"enrichr barplot ({_gs_lib}) 已保存: {_fig_path}")
else:
    print("无 enrichr 结果，跳过可视化")

## 2. decoupler 通路活性评分 -- PROGENy + MSigDB Hallmark

与 enrichr（基于预定义的 DEG 阈值）不同，decoupler 直接在表达矩阵上
计算通路活性分数——不需要先决定「哪些基因是显著的」。

**为什么用 ULM（Univariate Linear Model）？** 它是 decoupler 推荐的首选方法，
计算速度快（线性回归），结果可解释（每个基因对通路分数的贡献是线性的），
同时控制批次效应（通过 `sample_col` 在模型中加入样本协变量）。
相比 GSEA，ULM 不需要基因排序；相比 ORA，不依赖阈值。

### 通路数据库选择

- **PROGENy**（14 条信号通路）：覆盖核心癌症信号通路（EGFR/MAPK/PI3K/p53/TNFa/NFkB/
  JAK-STAT/Hippo/TGFb/...），是 pathway activity inference 的 gold standard。
  基因-通路权重基于公开扰动实验（perturbation experiments）数据训练。
- **MSigDB Hallmark**（50 条标志性通路）：覆盖更广泛的生物学过程
  （炎症、EMT、凋亡、缺氧、血管生成、代谢等），基因集源自 MSigDB，
  经过去冗余处理，每条 Hallmark 代表一个明确的生物学主题。

**为什么两个库都用？** PROGENy 专注信号通路且有权重信息（TF target 的置信度），
Hallmark 覆盖更广的生物学过程。两者互补——PROGENy 告诉你「哪条信号激活」，
Hallmark 告诉你「发生了哪个生物学过程」。

In [ ]:
# decoupler: 获取通路基因集。
# `dc.op.progeny()` 和 `dc.op.hallmark()` 从 decoupler 内置数据库获取，
# 返回 DataFrame，列为 source/target/weight。

_decoupler_results = {}  # resource_name -> activity DataFrame (clusters x pathways)
_decoupler_available = False

if _network_available:
    try:
        # 获取 PROGENy 通路基因集（14 条核心信号通路）
        _progeny_net = dc.op.progeny(organism="human", top=500)
        print(f"PROGENy: {_progeny_net['source'].nunique()} 通路, "
              f"{len(_progeny_net)} 基因-通路对")

        # 获取 MSigDB Hallmark 通路基因集（50 条标志性通路）
        _hallmark_net = dc.op.hallmark(organism="human")
        print(f"Hallmark: {_hallmark_net['source'].nunique()} 通路, "
              f"{len(_hallmark_net)} 基因-通路对")

        _decoupler_available = True
    except Exception as _e:
        print(f"decoupler 通路数据库获取失败: {type(_e).__name__}: {_e}")
        print("decoupler 通路活性评分将跳过")
else:
    print("网络不可用，decoupler 在线通路数据库获取跳过")
    print("如需离线使用，可预先下载 GMT 文件并用 dc.pp.read_gmt() 加载")

In [ ]:
# decoupler 通路活性评分：对每个细胞用 ULM 计算通路分数，
# 然后按分组列聚合（均值），得到每个簇-通路的平均活性。
#
# 为什么在 counts layer 上打分？decoupler 方法（ulm/mlm/gsva）
# 设计输入为 raw counts 或 log-normalized counts。
# 本基线数据 `counts` layer 含原始整数计数，适合 decoupler。

if _decoupler_available:
    # 选择用于 decoupler 的矩阵（优先 counts layer，其次 adata.X）
    if "counts" in adata.layers:
        _dc_data = adata.layers["counts"]
        print("decoupler 使用 counts layer（原始计数）")
    else:
        _dc_data = adata.X
        print("decoupler 使用 adata.X（无 counts layer）")

    # 构建临时 AnnData 用于 decoupler 计算。
    # 注意：X 使用 view 而非 copy，避免 count 矩阵峰值内存翻倍；
    # >50k 细胞时应先 subset 再传入。
    # var 仅需 var_names 用于 decoupler 基因匹配，不需要 var 列信息。
    import anndata as _ad
    _adata_dc = _ad.AnnData(
        X=_dc_data,
        obs=adata.obs[[_group_col]].copy(),
        var=pd.DataFrame(index=adata.var_names),
    )
    # 预留：多数据集 merge 后启用 sample_col
    if "sample_id" in adata.obs.columns:
        _adata_dc.obs["sample_id"] = adata.obs["sample_id"]

    for _res_name, _net in [("progeny", _progeny_net), ("hallmark", _hallmark_net)]:
        print(f"\n--- {_res_name} 通路活性评分 ({DECOUPLER_METHOD}) ---")
        try:
            # 运行 decoupler 通路评分
            _method_fn = getattr(dc.mt, DECOUPLER_METHOD)
            _method_fn(
                _adata_dc,
                net=_net,
                verbose=False,
            )

            # 结果在 obsm 中：score_{method} 和 padj_{method}
            # decoupler 2.1.6 的 naming convention 是 "score_*" 而非 "*_estimate"
            _est_key = f"score_{DECOUPLER_METHOD}"
            if _est_key not in _adata_dc.obsm:
                print(f"  WARNING: obsm key '{_est_key}' 未生成，跳过")
                continue

            # 按分组列聚合：每簇每通路的平均活性
            _act_df = _adata_dc.obsm[_est_key].copy()
            _act_df.index = _adata_dc.obs[_group_col].values
            _act_mean = _act_df.groupby(level=0).mean()
            print(f"  通路活性矩阵: {_act_mean.shape[1]} 通路 x {_act_mean.shape[0]} 簇")

            # 保存到 adata.obsm（细胞级结果）
            _cell_key = f"pathway_{_res_name}_{DECOUPLER_METHOD}"
            adata.obsm[_cell_key] = _adata_dc.obsm[_est_key].copy()

            _decoupler_results[_res_name] = _act_mean
            _csv = f"results/tables/stage7_pathway_decoupler_{_res_name}.csv"
            _act_mean.to_csv(_csv)
            print(f"  已导出: {_csv}")
        except Exception as _e:
            print(f"  {_res_name} 评分失败: {type(_e).__name__}: {_e}")

    del _adata_dc
    print(f"\ndecoupler 通路活性评分完成: {len(_decoupler_results)} 组结果")
else:
    print("decoupler 不可用，跳过通路活性评分")

### decoupler 可视化 -- 簇 x 通路 heatmap

对每簇的 decoupler 通路活性画热图。
**为什么用热图？** 簇 x 通路矩阵本质是二维数据，热图能同时展示簇的
通路活性谱（「这个簇哪几条通路高」）和通路跨簇分布（「这条通路在哪些簇高」）。
这对 PI 判断每个簇的功能偏性非常有价值——例如「簇 3 高 NFkB + TNFa +
炎症 Hallmark」很可能是免疫细胞。

In [ ]:
# decoupler 热图: 簇 x 通路活性。
# 选择每库的 top 可变通路（方差最大），避免图过大。
if _decoupler_results:
    for _res_name, _act_mean in _decoupler_results.items():
        # 选 top 可变通路
        _var = _act_mean.var().sort_values(ascending=False)
        _top_paths = _var.head(min(N_TOP_PATHWAYS_PLOT * 2, len(_var))).index.tolist()
        _plot_mat = _act_mean[_top_paths]

        fig, _ax = plt.subplots(figsize=(max(8, len(_top_paths) * 0.35),
                                        max(3, len(_plot_mat) * 0.3)))
        sns.heatmap(
            _plot_mat,
            cmap="RdBu_r",
            center=0,
            annot=False,
            fmt=".2f",
            xticklabels=True,
            yticklabels=True,
            ax=_ax,
        )
        _ax.set_title(f"Pathway Activity ({_res_name}) -- {DECOUPLER_METHOD}")
        _ax.set_xlabel("Pathway")
        _ax.set_ylabel("Cluster")
        plt.xticks(rotation=45, ha="right", fontsize=7)
        plt.tight_layout()
        _fig_path = f"results/figures/stage7_pathway_decoupler_heatmap_{_res_name}.png"
        fig.savefig(_fig_path, dpi=200, bbox_inches="tight")
        plt.close("all")
        print(f"decoupler heatmap ({_res_name}) 已保存: {_fig_path}")
else:
    print("无 decoupler 结果，跳过可视化")

## 结果保存

- enrichr 汇总表 -> `results/tables/stage7_pathway_enrichr_all.csv`
- decoupler 通路活性 -> `results/tables/stage7_pathway_decoupler_progeny.csv` / `hallmark.csv`
- 运行元数据 -> `adata.uns["stage7_pathway_v1"]`
- 产出 h5ad -> `OUTPUT_PATH`

In [ ]:
# 写入运行元数据到 adata.uns。
import datetime as _dt

_stage7_pathway_uns = {
    "group_col": _group_col,
    "n_genes_per_cluster": N_TOP_GENES_PER_CLUSTER,
    "network_available": _network_available,
    "enrichr": {
        "gene_sets": GSEAPY_GENE_SETS,
        "pval_cutoff": GSEAPY_PVAL_CUTOFF,
        "n_clusters_queried": len(_clusters_to_run) if _network_available else 0,
        "n_results": len(_enrichr_results),
        "csv": "results/tables/stage7_pathway_enrichr_all.csv",
    } if _network_available else {"skipped": True, "reason": "no network"},
    "decoupler": {
        "method": DECOUPLER_METHOD,
        "resources": list(_decoupler_results.keys()),
        "available": _decoupler_available,
    },
    "timestamp": _dt.datetime.now().isoformat(),
}
adata.uns["stage7_pathway_v1"] = _stage7_pathway_uns
print("运行元数据已写入 adata.uns['stage7_pathway_v1']")
print(f"  group_col: {_group_col}")
print(f"  network_available: {_network_available}")
print(f"  enrichr 结果组数: {len(_enrichr_results)}")
print(f"  decoupler 结果组数: {len(_decoupler_results)}")

In [ ]:
# 内存自检——确保 X 没有被误转为 dense。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")